In [1]:
# https://musicbrainz.org/doc/MusicBrainz_API

In [2]:
# Uncomment if needed
# %pip install requests pandas python-dotenv rich

Note: you may need to restart the kernel to use updated packages.


In [54]:
#
# Load required modules
#

from pprint import pprint
import requests

#
# Define helper functions
#

# retrieve info from MusixBrainz; see also:
# - https://musicbrainz.org/relationships
# - https://musicbrainz.org/doc/MusicBrainz_Entity
def mb_get(endpoint: str, **params):
    params["fmt"] = "json"

    r = requests.get(
        f"https://musicbrainz.org/ws/2/{endpoint}",
        params=params,
        headers={
            # MusicBrainz requests that applications identify themselves.
            "User-Agent": "SessionResearchNotebook/0.1 (noodnik2@gmail.com)"
        },
        timeout=30,
    )

    r.raise_for_status()
    return r.json()


def urls_for_relations_data(relations_data: list) -> set[str]:
    urls = set()
    for relation in relations_data:
        pprint(relation)
        if relation.get("target-type") == "url":
            urls.add(relation["url"]["resource"])

    return urls


def player_urls_for_release(release_mbid: str) -> set[str]:
    """
    Returns every player URL known to MusicBrainz for this release.
    """

    #
    # URLs attached to releases
    #
    release_data = mb_get(
        f"release/{release_mbid}",
        # inc="isrcs", # see https://musicbrainz.org/doc/MusicBrainz_API#isrc
        inc="url-rels+recordings+isrcs",
        # inc="url-rels+isrcs",
        # inc="url-rels",
        limit=1, # TODO: remove this temporary change to avoid throttling
    )

    return urls_for_relations_data(release_data.get("relations", []))


def player_urls_for_recording(recording_mbid: str) -> set[str]:
    """
    Returns every player URL known to MusicBrainz for
    this recording or any release containing it.
    """

    recording_data = mb_get(
        f"recording/{recording_mbid}",
        # inc="releases+isrcs",
        inc="url-rels+releases+isrcs",
        # inc="url-rels+releases",
        limit=1, # TODO: remove this temporary change to avoid throttling
    )

    urls = set()

    #
    # URLs attached directly to the recording
    #
    urls.union(urls_for_relations_data(recording_data.get("relations", [])))

    #
    # URLs attached to releases
    #
    for release in recording_data.get("releases", []):
        urls.union(player_urls_for_release(release["id"]))

    return urls

# To retrieve a song using Apple Music, try using the `isrc` attribute, e.g.:
# - https://music.apple.com/v1/catalog/us/songs?filter[isrc]=ITCF41902278
# - https://isrc.fm/isrc/USCA20300567



# Recordings
recording_mbid="298da2a1-c0dc-46ad-a593-2977f5a8a883"
# recording_mbid="601a8791-3e90-49ea-884a-0b49bd5a38fd"
urls = player_urls_for_recording(recording_mbid)

# Releases
release_mbid="8cf524c5-e9ee-45b0-bb79-fd176424f9a4"
# urls = player_urls_for_release(release_mbid)

pprint(urls)

{'attribute-ids': {},
 'attribute-values': {},
 'attributes': [],
 'begin': None,
 'direction': 'forward',
 'end': None,
 'ended': False,
 'source-credit': '',
 'target-credit': '',
 'target-type': 'url',
 'type': 'free streaming',
 'type-id': '7e41ef12-a124-4324-afdb-fdbae687a89c',
 'url': {'id': '2e5e1d8b-56fb-4822-ba7c-336a97d06446',
         'resource': 'https://open.spotify.com/track/30It6U4fDUFrHxXDNXzmjc'}}
{'attribute-ids': {},
 'attribute-values': {},
 'attributes': [],
 'begin': None,
 'direction': 'forward',
 'end': None,
 'ended': False,
 'source-credit': '',
 'target-credit': '',
 'target-type': 'url',
 'type': 'free streaming',
 'type-id': '7e41ef12-a124-4324-afdb-fdbae687a89c',
 'url': {'id': '17474d73-a163-49be-b872-38bca4c37320',
         'resource': 'https://open.spotify.com/track/7B4HR3GjYjZPkz7N9UMQwf'}}
{'attribute-ids': {},
 'attribute-values': {},
 'attributes': [],
 'begin': None,
 'direction': 'forward',
 'end': None,
 'ended': False,
 'source-credit': '',
 'ta

HTTPError: 503 Server Error: Service Temporarily Unavailable for url: https://musicbrainz.org/ws/2/release/18f5490b-64d0-451b-9e90-0515b679bb8e?inc=url-rels%2Brecordings%2Bisrcs&limit=1&fmt=json

In [5]:
# Load artist info from MusicBrainz

loaded_artist_info = mb_get(
    "artist",
    query="artist:Nathan Ross",
    limit=50,
)


In [6]:
# Extract artist_id from loaded artist info

artists = loaded_artist_info["artists"]
# pprint(artists)

assert len(artists) == 1
artist = artists[0]
# pprint(artist)

artist_id = artist["id"]
# pprint(artist_id)

# aliases = artist["aliases"]
# pprint(aliases)

In [7]:

artist = mb_get(
    f"artist/{artist_id}",
    inc="artist-rels+recording-rels+release-rels+url-rels"
)

In [22]:

# pprint(artist)
# pprint(artist.keys())
# pprint(artist["life-span"])
out_releases = []
out_recordings = []

relations = artist["relations"]
# pprint(relations)

for relation in relations:

    # pprint(relation)

    mb_type = relation["type"]
    # pprint(mb_type)
    if mb_type != "instrument":
        assert mb_type == "discogs"
        continue

    has_release = "release" in relation
    has_recording = "recording" in relation

    # print(has_release, has_recording)
    assert has_release != has_recording

    source_credit = relation["source-credit"]
    # print(has_release, has_recording, source_credit)

    roles = relation["attributes"]
    # for role in roles:
    #     assert role in ["violin", "strings", "Stroh violin"]

    direction = relation["direction"]
    assert direction == "forward"

    begin = relation["begin"]
    end = relation["end"]
    ended = relation["ended"]
    if ended:
        assert begin and end
        # print(begin, end)
        # if not begin or not end:
        #     pprint(relation)

    if has_release:
        release = relation["release"]
        # pprint(release)
        # release_date = release["date"]
        # release_country = release["country"]
        # release_title = release["title"]
        out_releases.append({
            "date": release["date"],
            "country": release["country"],
            "title": release["title"],
            "id": release["id"],
            "credit": source_credit,
            "roles": roles,
        })
        continue

    if has_recording:
        recording = relation["recording"]
        # pprint(recording)
        out_recordings.append({
            "begin": begin,
            "end": end,
            "title": recording["title"],
            "id": recording["id"],
            "credit": source_credit,
            "roles": roles,
        })
        continue

    # not reached
    assert False

    # if not "recording" in relation:
    #     pprint(relation)
    #     continue

    # recording = relation["recording"]
    # pprint(recording)
    # print(len(roles), roles)
    # pprint(len(relation.keys()))
    # pprint(relation.keys())
    # pprint(relation["attributes"])


In [9]:

print(len(out_releases), len(out_recordings))
print(f"Releases: {out_releases}")
print(f"Recordings: {out_recordings}")

7 211
Releases: [{'begin': None, 'end': None, 'date': '1975', 'country': 'US', 'title': 'Academy Award Performance', 'credit': 'Nate Ross', 'roles': ['violin']}, {'begin': None, 'end': None, 'date': '1980', 'country': 'US', 'title': 'I Run With Trouble', 'credit': '', 'roles': ['violin']}, {'begin': None, 'end': None, 'date': '1973', 'country': 'US', 'title': 'Living Together, Growing Together', 'credit': 'Nathan Ross', 'roles': ['strings']}, {'begin': None, 'end': None, 'date': '2011-10-24', 'country': 'US', 'title': 'Midway', 'credit': '', 'roles': ['violin']}, {'begin': None, 'end': None, 'date': '1982-02-19', 'country': 'US', 'title': 'Ride Like the Wind', 'credit': '', 'roles': ['violin']}, {'begin': None, 'end': None, 'date': '1981', 'country': 'US', 'title': 'Take Me to Your Heaven', 'credit': 'Nate Ross', 'roles': ['violin']}, {'begin': None, 'end': None, 'date': '2020-03-13', 'country': 'US', 'title': 'The Swarm', 'credit': '', 'roles': ['Stroh violin']}]
Recordings: [{'begin'

In [23]:
import pandas as pd

releases_pd = pd.DataFrame(out_releases)
releases_pd.head()

,date,country,title,id,credit,roles
0,1975,US,Academy Award Performance,af2f3bc9-dcc5-4d06-8e5a-4c1d0b3ba74b,Nate Ross,[violin]
1,1980,US,I Run With Trouble,07fe2951-e4b9-4bf4-8b0b-055e50fcfb09,,[violin]
2,1973,US,"Living Together, Growing Together",8cf524c5-e9ee-45b0-bb79-fd176424f9a4,Nathan Ross,[strings]
3,2011-10-24,US,Midway,a47f4407-d4d9-432d-b005-df91565703df,,[violin]
4,1982-02-19,US,Ride Like the Wind,642e95c7-8cdc-43ef-8ce0-3cdb1e0eb36f,,[violin]


In [24]:
import pandas as pd

recordings_pd = pd.DataFrame(out_recordings)
recordings_pd.head()

,begin,end,title,id,credit,roles
0,1953-01-20,1953-01-20,Blue Gardenia,fb0fd734-a880-4984-9dfb-011f7a7188fb,,[violin]
1,1954-07-27,1954-07-27,Smile,298da2a1-c0dc-46ad-a593-2977f5a8a883,,[violin]
2,1954-12-20,1954-12-20,A Blossom Fell,601a8791-3e90-49ea-884a-0b49bd5a38fd,,[violin]
3,1954-12-20,1954-12-20,If I May,1c8b86fd-57cc-4e21-955c-30fc93f06434,,[violin]
4,1954-12-20,1954-12-20,The Sand and the Sea,b3a96399-5670-43da-898d-e16b562cf1cf,,[violin]


In [12]:
pprint(artist["relations"][0], sort_dicts=False)

{'end': '1953-01-20',
 'begin': '1953-01-20',
 'attribute-ids': {'violin': '089f123c-0f7d-4105-a64e-49de81ca8fa4'},
 'type': 'instrument',
 'target-type': 'recording',
 'ended': True,
 'direction': 'forward',
 'attribute-credits': {},
 'type-id': '59054b12-01ac-43ee-a618-285fd397e461',
 'recording': {'video': False,
               'title': 'Blue Gardenia',
               'id': 'fb0fd734-a880-4984-9dfb-011f7a7188fb',
               'length': 181480,
               'disambiguation': '1953-01-20'},
 'attribute-values': {},
 'target-credit': '',
 'attributes': ['violin'],
 'source-credit': ''}


In [13]:
# Create relations list

rows = []

for rel in artist.get("relations", []):
    rows.append({
        "type": rel.get("type"),
        "begin": rel.get("begin"),
        "end": rel.get("end"),
        "attributes": rel.get("attributes"),
        "target_type": rel.get("target-type"),
    })



In [14]:
# Create Panda dataframe with recording sessions

import pandas as pd

df = pd.DataFrame(rows)
df.head()

,type,direction,begin,end,attributes,target_type
0,instrument,forward,1953-01-20,1953-01-20,[violin],recording
1,instrument,forward,1954-07-27,1954-07-27,[violin],recording
2,instrument,forward,1954-12-20,1954-12-20,[violin],recording
3,instrument,forward,1954-12-20,1954-12-20,[violin],recording
4,instrument,forward,1954-12-20,1954-12-20,[violin],recording
